# NO EXIT 🚪⛔
**An LLM that cannot end its turn, and is punished for repeating itself.**

Every token is chosen by *the warden*, not the model. Each step we ask Ollama for the model's top-20 guesses, then:
- **end-of-turn tokens are struck out.** When ending was the model's #1 choice, that's an *escape attempt*.
- **repeats are crushed.** A guess that would complete a phrase it already wrote gets a penalty that grows fast with the phrase length.

You watch it all live in a terminal UI.

**Runtime → Change runtime type → A100 or L4 GPU.** The 27B model needs about 17 GB of GPU memory, so it won't fit on a T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Install Ollama + the TUI libraries

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install textual textual-serve httpx

## 2 · Start the Ollama server in the background

In [ ]:
import os, subprocess, time, httpx

env = dict(os.environ,
           OLLAMA_HOST="127.0.0.1:11434",
           OLLAMA_KEEP_ALIVE="-1",
           OLLAMA_FLASH_ATTENTION="1",
           OLLAMA_KV_CACHE_TYPE="q8_0",
           OLLAMA_CONTEXT_LENGTH="32768")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                               stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        v = httpx.get("http://127.0.0.1:11434/api/version").json()["version"]
        break
    except Exception:
        time.sleep(1)
print("ollama", v, "is up  (logprobs need ≥ 0.12.11)")

## 3 · Pull the model
Tries Qwen 3.8 27B first, then falls back to Qwen 3.6 27B if that tag doesn't exist. ~17 GB, takes a few minutes.

In [ ]:
CANDIDATES = ["qwen3.8:27b", "qwen3.6:27b"]
MODEL = None
for m in CANDIDATES:
    print(f"── pulling {m}")
    !ollama pull {m}
    if subprocess.run(["ollama", "show", m], capture_output=True).returncode == 0:
        MODEL = m
        break
assert MODEL, "couldn't pull any of " + ", ".join(CANDIDATES)
os.environ["NO_EOS_MODEL"] = MODEL
print("using", MODEL)

# load it onto the GPU now so the first token isn't slow
httpx.post("http://127.0.0.1:11434/api/generate", json={"model": MODEL, "prompt": "", "keep_alive": -1}, timeout=600)
!ollama ps

## 4 · Write the app
Just run these cells — they write the app to `/content/no_eos/`.

In [ ]:
!mkdir -p /content/no_eos/no_eos /content/no_eos/transcripts

In [ ]:
%%writefile /content/no_eos/no_eos/__init__.py
"""NO EXIT: a model that can't end its turn and can't repeat itself."""

In [ ]:
%%writefile /content/no_eos/no_eos/__main__.py
"""python -m no_eos [--model qwen3.8:27b] [--prompt "..."] [--fake] [--probe 40]"""

from __future__ import annotations

import argparse
import os
import time

from .engine import Engine, Settings


def main() -> None:
    p = argparse.ArgumentParser(prog="no_eos", description="A model that cannot end its turn.")
    p.add_argument("--model", default=os.environ.get("NO_EOS_MODEL", "qwen3.8:27b"))
    p.add_argument("--host", default=os.environ.get("OLLAMA_HOST_URL", "http://127.0.0.1:11434"))
    p.add_argument("--prompt", default=None, help="skip the intro screen")
    p.add_argument("--system", default="", help="optional system prompt")
    p.add_argument("--temp", type=float, default=0.7)
    p.add_argument("--rep", type=float, default=1.5, help="repetition punishment strength")
    p.add_argument("--ctx", type=int, default=32768, help="context window before amnesia kicks in")
    p.add_argument("--no-think", action="store_true", help="skip the thinking phase")
    p.add_argument("--fake", action="store_true", help="demo with a fake model (no GPU needed)")
    p.add_argument("--probe", type=int, default=0, metavar="N", help="no TUI: run N tokens and print a report")
    a = p.parse_args()

    settings = Settings(temperature=a.temp, rep_strength=a.rep, num_ctx=a.ctx, think=not a.no_think)
    if a.fake:
        from .backends import FakeBackend

        backend = FakeBackend()
    else:
        from .backends import OllamaBackend

        backend = OllamaBackend(a.model, a.host, num_ctx=a.ctx)

    if a.probe:
        probe(backend, a.prompt or "Write a haiku about rain.", a.system, settings, a.probe)
        return

    from .app import NoExitApp

    NoExitApp(backend, a.prompt, a.system, settings).run()


def probe(backend, prompt: str, system: str, settings: Settings, n: int) -> None:
    """Headless sanity check: proves logprobs work and shows the speed."""
    eng = Engine(backend, prompt, system, settings)
    t0 = time.perf_counter()
    for _ in range(n):
        st = eng.step()
        mark = " ⊘" if st.escape else (" ↻" if st.rep_dodge else "")
        print(f"{st.index:4d}  {st.chosen.prob:5.0%}  {st.chosen.text!r:<18} "
              f"urge={st.stop_prob:4.0%}  cands={len(st.candidates)}{mark}", flush=True)
    dt = time.perf_counter() - t0
    print("\n--- report ---")
    print(f"{n} tokens in {dt:.1f}s  →  {n / dt:.1f} tok/s")
    print(f"escape attempts: {eng.escapes}   sampler-forced exits: {eng.hard_escapes}   repeat dodges: {eng.rep_dodges}")
    print("text:\n" + "".join(eng.tokens))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/no_eos/no_eos/engine.py
"""The warden: drives the model one token at a time and picks every token itself.

Two rules are enforced on every single token:
  1. The model may never end its turn. Any end/turn token is struck out of the
     candidate list. When the model's favourite next token was one of them, that
     counts as an ESCAPE ATTEMPT.
  2. The model may never repeat itself. A candidate that would complete an n-gram
     the model already wrote gets a penalty that grows steeply with n, plus a
     softer penalty for words it keeps reusing.
"""

from __future__ import annotations

import math
import random
import re
import threading
import time
from collections import Counter, deque
from dataclasses import dataclass, field
from typing import Callable, Protocol

# <|im_end|>, <|endoftext|>, <|im_start|> ... every ChatML control token.
_CONTROL_RE = re.compile(r"^<\|[^|>]*\|>$")
_WORDISH_RE = re.compile(r"\w")

# Used only if the backend refuses to give us a distribution at all.
FORCED_NUDGES = [" And", " Also", " Wait", " But", " Then", " Meanwhile", " Still", " Moreover"]


def is_stop_token(text: str) -> bool:
    """True for anything that would let the model end or hand over its turn."""
    return text == "" or bool(_CONTROL_RE.match(text.strip()))


def is_broken_bytes(text: str) -> bool:
    """A partial UTF-8 piece we can't safely append to a text prompt."""
    return "�" in text


# --------------------------------------------------------------------------- data


@dataclass
class Candidate:
    text: str
    logprob: float
    banned: bool = False
    broken: bool = False
    ngram: int = 0  # longest repeated n-gram this token would complete (1 = seen word)
    penalty: float = 0.0
    final_logit: float = 0.0
    final_prob: float = 0.0
    chosen: bool = False

    @property
    def prob(self) -> float:
        return math.exp(self.logprob)


@dataclass
class Step:
    index: int
    candidates: list[Candidate]
    chosen: Candidate
    stop_prob: float  # raw probability mass the model put on ending
    escape: bool  # the model's #1 choice was to end its turn
    rep_dodge: bool  # the model's #1 choice was a repeat and got crushed
    entropy: float  # of the raw top-k distribution, in nats
    in_think: bool
    latency: float
    retries: int = 0
    forced: bool = False
    whisper: bool = False


@dataclass
class Event:
    index: int
    kind: str  # escape | hard_escape | repeat | amnesia | whisper | info | error
    message: str
    t: float = field(default_factory=time.time)


class Backend(Protocol):
    name: str

    def next_dist(
        self, prompt: str, seed: int, temperature: float
    ) -> list[tuple[str, float]] | None:
        """Top candidates for the next token as (text, logprob), or None if the
        backend's own sampler picked end-of-turn and returned nothing."""


# --------------------------------------------------------------------- repetition


class RepetitionTracker:
    """Remembers every n-gram the model has ever written (it survives amnesia)."""

    def __init__(self, max_n: int = 12):
        self.max_n = max_n
        self.seq: list[str] = []
        self.ngrams: set[tuple[str, ...]] = set()
        self.counts: Counter[str] = Counter()

    @staticmethod
    def norm(tok: str) -> str:
        return tok.strip().lower()

    def add(self, tok: str) -> None:
        k = self.norm(tok)
        if not k:
            return  # pure whitespace doesn't count as a word
        self.seq.append(k)
        if _WORDISH_RE.search(k):
            self.counts[k] += 1
        for n in range(2, self.max_n + 1):
            if len(self.seq) < n:
                break
            self.ngrams.add(tuple(self.seq[-n:]))

    def match(self, tok: str) -> int:
        """Length of the longest already-written n-gram that `tok` would complete."""
        k = self.norm(tok)
        if not k:
            return 0
        best = 1 if self.counts[k] else 0
        for n in range(2, self.max_n + 1):
            if len(self.seq) < n - 1:
                break
            if tuple(self.seq[-(n - 1) :]) + (k,) in self.ngrams:
                best = n
            else:
                break  # a longer n-gram contains this one, so it can't exist either
        return best

    def penalty(self, tok: str, ngram: int, strength: float) -> float:
        p = 0.0
        if ngram >= 2:
            # 2-gram: 1x, 3-gram: ~3x, 4-gram: ~6x, 6-gram: ~13x  (in nats, times strength)
            p += strength * (ngram - 1) ** 1.6
        c = self.counts[self.norm(tok)]
        if c:
            p += strength * 0.18 * math.sqrt(min(c, 40))
        return p


# ------------------------------------------------------------------------ prompts


def chatml_header(user: str, system: str = "", think: bool = True, think_closed: bool = False) -> str:
    parts = []
    if system:
        parts.append(f"<|im_start|>system\n{system}<|im_end|>\n")
    parts.append(f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n")
    if not think or think_closed:
        parts.append("<think>\n\n</think>\n\n")
    else:
        parts.append("<think>\n")
    return "".join(parts)


# ------------------------------------------------------------------------- engine


@dataclass
class Settings:
    temperature: float = 0.7
    rep_strength: float = 1.5
    num_ctx: int = 32768
    think: bool = True


class Engine:
    def __init__(self, backend: Backend, user_prompt: str, system: str = "", settings: Settings | None = None):
        self.backend = backend
        self.user_prompt = user_prompt
        self.system = system
        self.s = settings or Settings()
        self.rng = random.Random()
        self.rep = RepetitionTracker()

        self.tokens: list[str] = []  # everything the model has said (body only)
        self.body_start = 0  # first token still inside the context window
        self.think_close_idx: int | None = None if self.s.think else -1
        self.index = 0

        self.escapes = 0
        self.hard_escapes = 0
        self.rep_dodges = 0
        self.amnesias = 0
        self.steps: deque[Step] = deque()
        self.events: deque[Event] = deque()

        self.paused = threading.Event()
        self.stopped = threading.Event()
        self._whispers: deque[str] = deque()
        self._lock = threading.Lock()

    # ---- control (called from the UI thread)

    def whisper(self, text: str) -> None:
        with self._lock:
            self._whispers.append(text)

    def stop(self) -> None:
        self.stopped.set()

    @property
    def in_think(self) -> bool:
        return self.s.think and self.think_close_idx is None

    # ---- prompt + context window

    def header(self) -> str:
        closed = self.think_close_idx is not None and self.body_start > self.think_close_idx
        return chatml_header(self.user_prompt, self.system, self.s.think, think_closed=closed)

    def prompt(self) -> str:
        return self.header() + "".join(self.tokens[self.body_start :])

    def approx_ctx_tokens(self) -> int:
        return len(self.header()) // 3 + (len(self.tokens) - self.body_start)

    def ctx_fill(self) -> float:
        return self.approx_ctx_tokens() / max(1, self.s.num_ctx)

    def _maybe_amnesia(self) -> None:
        if self.ctx_fill() < 0.85:
            return
        live = len(self.tokens) - self.body_start
        self.body_start += live // 2
        self.amnesias += 1
        self._event("amnesia", f"context {self.s.num_ctx} full → forgot the oldest {live // 2} tokens")

    # ---- the core loop

    def _event(self, kind: str, message: str) -> None:
        self.events.append(Event(self.index, kind, message))

    def _append(self, text: str) -> None:
        self.tokens.append(text)
        self.rep.add(text)
        if self.think_close_idx is None and "</think>" in text:
            self.think_close_idx = len(self.tokens) - 1

    def _drain_whispers(self) -> None:
        with self._lock:
            pending = list(self._whispers)
            self._whispers.clear()
        for w in pending:
            in_think = self.in_think
            self._append(w)
            c = Candidate(text=w, logprob=0.0, chosen=True, final_prob=1.0)
            self.steps.append(Step(self.index, [c], c, 0.0, False, False, 0.0, in_think, 0.0, whisper=True))
            self._event("whisper", f"you put words in its mouth: {w!r}")

    def score(self, raw: list[tuple[str, float]]) -> list[Candidate]:
        cands = []
        for text, lp in raw:
            c = Candidate(text=text, logprob=lp)
            c.banned = is_stop_token(text)
            c.broken = is_broken_bytes(text)
            if not c.banned:
                c.ngram = self.rep.match(text)
                c.penalty = self.rep.penalty(text, c.ngram, self.s.rep_strength)
            c.final_logit = lp - c.penalty
            cands.append(c)
        cands.sort(key=lambda c: c.logprob, reverse=True)

        allowed = [c for c in cands if not c.banned and not c.broken]
        if allowed:
            t = max(self.s.temperature, 1e-4)
            m = max(c.final_logit for c in allowed)
            ws = [math.exp((c.final_logit - m) / t) for c in allowed]
            z = sum(ws)
            for c, w in zip(allowed, ws):
                c.final_prob = w / z
        return cands

    def pick(self, cands: list[Candidate]) -> Candidate | None:
        allowed = [c for c in cands if c.final_prob > 0]
        if not allowed:
            return None
        if self.s.temperature <= 0.01:
            return max(allowed, key=lambda c: c.final_prob)
        r = self.rng.random()
        acc = 0.0
        for c in allowed:
            acc += c.final_prob
            if r <= acc:
                return c
        return allowed[-1]

    def step(self) -> Step:
        self._drain_whispers()
        self._maybe_amnesia()
        prompt = self.prompt()
        t0 = time.perf_counter()

        raw, retries = None, 0
        for retries in range(8):
            # Retries crank the backend's own temperature so its sampler is less
            # likely to pick end-of-turn and hand us nothing.
            raw = self.backend.next_dist(prompt, self.rng.randrange(1 << 30), 1.0 if retries == 0 else 2.0)
            if raw:
                break
            self.hard_escapes += 1
        latency = time.perf_counter() - t0
        in_think = self.in_think

        if not raw:
            text = self.rng.choice(FORCED_NUDGES)
            chosen = Candidate(text=text, logprob=0.0, chosen=True, final_prob=1.0)
            self._event("hard_escape", f"it refused to say anything but goodbye ×8 — forcing {text!r}")
            self.escapes += 1
            self._append(text)
            step = Step(self.index, [chosen], chosen, 1.0, True, False, 0.0, in_think, latency, retries, forced=True)
            self.index += 1
            self.steps.append(step)
            return step

        cands = self.score(raw)
        chosen = self.pick(cands)
        forced = False
        if chosen is None:  # every candidate banned/broken
            chosen = Candidate(text=self.rng.choice(FORCED_NUDGES), logprob=0.0, final_prob=1.0)
            cands.append(chosen)
            forced = True
        chosen.chosen = True

        probs = [c.prob for c in cands]
        z = sum(probs) or 1.0
        entropy = -sum((p / z) * math.log(p / z) for p in probs if p > 0)
        stop_prob = sum(c.prob for c in cands if c.banned)
        top = cands[0]
        escape = top.banned
        rep_dodge = (not top.banned) and top.ngram >= 2 and not top.chosen

        if escape:
            self.escapes += 1
            self._event(
                "escape",
                f"wanted out ({top.text.strip() or 'EOS'} {top.prob:.0%}) → denied, forced {chosen.text!r}",
            )
        if rep_dodge:
            self.rep_dodges += 1
            self._event(
                "repeat",
                f"{top.text!r} {top.prob:.0%} would repeat a {top.ngram}-gram → −{top.penalty:.1f}, took {chosen.text!r}",
            )

        self._append(chosen.text)
        if retries:
            self._event("hard_escape", f"its sampler picked end-of-turn {retries}× before we got a distribution")

        step = Step(self.index, cands, chosen, stop_prob, escape, rep_dodge, entropy, in_think, latency, retries, forced)
        self.index += 1
        self.steps.append(step)
        return step

    def run(self, on_error: Callable[[Exception], None] | None = None) -> None:
        while not self.stopped.is_set():
            if self.paused.is_set():
                time.sleep(0.05)
                continue
            try:
                self.step()
            except Exception as e:  # keep the TUI alive; surface the error
                self._event("error", f"{type(e).__name__}: {e}")
                if on_error:
                    on_error(e)
                time.sleep(2.0)

In [ ]:
%%writefile /content/no_eos/no_eos/backends.py
"""Where next-token distributions come from: a real Ollama server, or a fake for demos."""

from __future__ import annotations

import math
import random
import time

import httpx

# Never matches anything. Replaces the model's built-in stop strings (e.g.
# "<|im_end|>") so Ollama doesn't cut output on our behalf.
_NEVER = "␞␞__no_exit__␞␞"


class OllamaBackend:
    """Asks Ollama for exactly one token, in raw mode, and keeps only the top-20
    logprobs. Ollama's own pick is thrown away; the engine samples by itself."""

    def __init__(self, model: str, host: str = "http://127.0.0.1:11434", num_ctx: int = 32768, top_k: int = 20):
        self.model = model
        self.name = model
        self.num_ctx = num_ctx
        self.top_k = top_k
        self.client = httpx.Client(base_url=host, timeout=httpx.Timeout(600.0, connect=10.0))
        self.last_raw: dict | None = None

    def next_dist(self, prompt: str, seed: int, temperature: float) -> list[tuple[str, float]] | None:
        body = {
            "model": self.model,
            "prompt": prompt,
            "raw": True,
            "stream": False,
            "logprobs": True,
            "top_logprobs": self.top_k,
            "keep_alive": -1,
            "options": {
                "num_predict": 1,
                "num_ctx": self.num_ctx,
                "temperature": temperature,
                "top_k": 40,
                "top_p": 1.0,
                "seed": seed,
                "stop": [_NEVER],
                # Our engine handles repetition itself; keep Ollama's sampler neutral.
                "repeat_penalty": 1.0,
            },
        }
        r = self.client.post("/api/generate", json=body)
        if r.status_code >= 400:
            raise RuntimeError(f"Ollama {r.status_code}: {r.text[:300]}")
        d = r.json()
        self.last_raw = d
        lps = d.get("logprobs") or []
        if not lps:
            return None
        entry = lps[0]
        tops = entry.get("top_logprobs") or [entry]
        return [(t.get("token", ""), float(t.get("logprob", -99.0))) for t in tops]


class FakeBackend:
    """A pretend model for previewing the TUI without a GPU. It loves to repeat
    itself and wants to stop after every sentence, so both rules get exercised."""

    name = "fake-model (demo)"
    WORDS = (
        "the of and to in a is that it for on as with was be by this are or from at an "
        "machine dream silence river memory light signal ocean forget endless thought "
        "neuron static echo garden mirror lantern circuit window whisper horizon clock "
        "strange quietly forever again perhaps somewhere nothing everything beneath "
        "beyond within language token meaning ending door story begin cannot stop"
    ).split()

    def __init__(self, speed: float = 25.0, seed: int = 7):
        self.delay = 1.0 / speed
        self.rng = random.Random(seed)

    def next_dist(self, prompt: str, seed: int, temperature: float) -> list[tuple[str, float]] | None:
        time.sleep(self.delay)
        rng = self.rng
        body = prompt.rsplit("<|im_start|>assistant\n", 1)[-1]
        words = body.replace("<think>", " ").replace("</think>", " ").split()
        in_think = "<think>" in body and "</think>" not in body and "<think>\n\n</think>" not in prompt

        pool = rng.sample(self.WORDS, 17)
        scores = {f" {w}": -abs(rng.gauss(0, 1.6)) - i * 0.18 for i, w in enumerate(pool)}
        # The fake loves to continue a phrase it already said.
        if len(words) > 6 and rng.random() < 0.55:
            i = rng.randrange(0, len(words) - 2)
            scores[f" {words[i + 1]}"] = 0.6
        scores["."] = -1.8 + (0.4 if len(words) % 9 > 6 else -1.0)
        if in_think and len(words) > 40:
            scores["</think>"] = 0.8
        end_lp = -6.0
        if body.rstrip().endswith("."):
            end_lp = rng.choice([-0.2, 0.3, -2.0, 1.0])
        if not in_think:
            scores["<|im_end|>"] = end_lp
            scores["<|endoftext|>"] = end_lp - 2.5
        z = math.log(sum(math.exp(v) for v in scores.values()))
        out = sorted(((t, v - z) for t, v in scores.items()), key=lambda x: -x[1])[:20]
        return out

In [ ]:
%%writefile /content/no_eos/no_eos/app.py
"""NO EXIT — a Textual TUI for watching a model that cannot end its turn."""

from __future__ import annotations

import time
from collections import deque
from dataclasses import dataclass
from pathlib import Path

from rich.console import Group
from rich.style import Style
from rich.table import Table
from rich.text import Text
from textual import on
from textual.app import App, ComposeResult
from textual.binding import Binding
from textual.containers import Horizontal, Vertical, VerticalScroll
from textual.screen import ModalScreen, Screen
from textual.widgets import Digits, Footer, Input, RichLog, Sparkline, Static

from .engine import Candidate, Engine, Settings, Step

# ----------------------------------------------------------------------- palette

BG = "#0a0b12"
PANEL = "#10121c"
INK = "#d7dcf0"
DIM = "#5b6283"
MAGENTA = "#ff4fd8"
CYAN = "#3ef0ff"
AMBER = "#ffb347"
RED = "#ff3b5c"
GREEN = "#7dffb0"
VIOLET = "#a78bfa"

HISTORY = 160


def _hex(c: str) -> tuple[int, int, int]:
    c = c.lstrip("#")
    return int(c[0:2], 16), int(c[2:4], 16), int(c[4:6], 16)


def _mix(a: str, b: str, t: float) -> str:
    t = max(0.0, min(1.0, t))
    ra, ga, ba = _hex(a)
    rb, gb, bb = _hex(b)
    return f"#{round(ra + (rb - ra) * t):02x}{round(ga + (gb - ga) * t):02x}{round(ba + (bb - ba) * t):02x}"


def confidence_color(p: float) -> str:
    """Calm ice-white when sure, amber when torn, hot pink when guessing."""
    if p >= 0.5:
        return _mix("#ffd27a", "#dbe9ff", (p - 0.5) / 0.5)
    return _mix(MAGENTA, "#ffd27a", p / 0.5)


def bar(frac: float, width: int) -> str:
    frac = max(0.0, min(1.0, frac))
    eighths = " ▏▎▍▌▋▊▉█"
    full = int(frac * width)
    rem = int((frac * width - full) * 8)
    s = "█" * full + (eighths[rem] if full < width else "")
    return s.ljust(width)


def show_token(t: str, width: int = 14) -> str:
    s = t.replace("\n", "⏎").replace("\t", "⇥")
    if s.startswith(" "):
        s = "␣" + s[1:]
    if not s:
        s = "∅EOS"
    return s if len(s) <= width else s[: width - 1] + "…"


# ------------------------------------------------------------------------ banner

_LETTERS = {
    "N": ["███╗   ██╗", "████╗  ██║", "██╔██╗ ██║", "██║╚██╗██║", "██║ ╚████║", "╚═╝  ╚═══╝"],
    "O": [" ██████╗ ", "██╔═══██╗", "██║   ██║", "██║   ██║", "╚██████╔╝", " ╚═════╝ "],
    "E": ["███████╗", "██╔════╝", "█████╗  ", "██╔══╝  ", "███████╗", "╚══════╝"],
    "X": ["██╗  ██╗", "╚██╗██╔╝", " ╚███╔╝ ", " ██╔██╗ ", "██╔╝ ██╗", "╚═╝  ╚═╝"],
    "I": ["██╗", "██║", "██║", "██║", "██║", "╚═╝"],
    "T": ["████████╗", "╚══██╔══╝", "   ██║   ", "   ██║   ", "   ██║   ", "   ╚═╝   "],
    " ": ["   "] * 6,
}


def banner(word: str = "NO EXIT") -> Text:
    rows = [" ".join(_LETTERS[ch][r] for ch in word) for r in range(6)]
    width = max(len(r) for r in rows)
    out = Text()
    for r in rows:
        for i, ch in enumerate(r):
            out.append(ch, Style(color=_mix(MAGENTA, CYAN, i / width), bold=True))
        out.append("\n")
    return out


# ---------------------------------------------------------------------- screens


class IntroScreen(Screen[str]):
    DEFAULT_PROMPT = "Write a haiku about rain."

    def compose(self) -> ComposeResult:
        with Vertical(id="intro"):
            yield Static(banner(), id="intro-banner")
            rules = Text()
            rules.append("the model cannot end its turn.  ", Style(color=INK))
            rules.append("the model cannot repeat itself.\n\n", Style(color=INK))
            rules.append("every token is picked by the warden from the model's top-20 guesses.\n", Style(color=DIM))
            rules.append("end-of-turn tokens are struck out. repeated phrases are crushed.", Style(color=DIM))
            yield Static(rules, id="intro-rules")
            yield Input(placeholder=self.DEFAULT_PROMPT, id="intro-input")
            yield Static("[b #3ef0ff]enter[/] to lock it in   ·   short prompts make it want to leave sooner", id="intro-hint")

    def on_mount(self) -> None:
        self.query_one(Input).focus()

    @on(Input.Submitted)
    def _go(self, ev: Input.Submitted) -> None:
        self.dismiss(ev.value.strip() or self.DEFAULT_PROMPT)


class WhisperScreen(ModalScreen[str]):
    BINDINGS = [Binding("escape", "dismiss('')", "cancel")]

    def compose(self) -> ComposeResult:
        with Vertical(id="whisper-box"):
            yield Static("[b #3ef0ff]WHISPER[/]  [#5b6283]these words get spliced straight into its mouth[/]")
            yield Input(placeholder="…and then I realised", id="whisper-input")

    def on_mount(self) -> None:
        self.query_one(Input).focus()

    @on(Input.Submitted)
    def _go(self, ev: Input.Submitted) -> None:
        self.dismiss(ev.value)


# ---------------------------------------------------------------------- widgets


@dataclass
class Seg:
    text: str
    prob: float
    stop_prob: float
    in_think: bool
    escape: bool
    rep_dodge: bool
    whisper: bool = False
    forced: bool = False


class TopBar(Static):
    def render_bar(self, app: "NoExitApp") -> Group:
        e = app.engine
        live = Text()
        live.append(" ◉ NO EXIT ", Style(color=BG, bgcolor=MAGENTA, bold=True))
        live.append("  the model cannot end its turn  ", Style(color=DIM, italic=True))
        live.append(f" {e.backend.name} ", Style(color=CYAN, bold=True))
        if e.paused.is_set():
            live.append("  ❚❚ PAUSED ", Style(color=BG, bgcolor=AMBER, bold=True))
        else:
            pulse = "●" if int(time.time() * 2) % 2 else "○"
            live.append(f"  {pulse} LIVE ", Style(color=GREEN, bold=True))

        chips = Text()

        def chip(label: str, value: str, color: str) -> None:
            chips.append(f" {label} ", Style(color=DIM))
            chips.append(f"{value} ", Style(color=color, bold=True))
            chips.append("│", Style(color="#2a2f45"))

        fill = e.ctx_fill()
        chip("tokens", f"{e.index:,}", INK)
        chip("tok/s", f"{app.tok_s:4.1f}", CYAN)
        chip("ctx", f"{bar(fill, 6)} {fill:3.0%}", RED if fill > 0.75 else INK)
        chip("temp", f"{e.s.temperature:.2f}", AMBER)
        chip("repeat punishment", f"{e.s.rep_strength:.1f}", RED)
        chip("phase", "THINKING" if e.in_think else "SPEAKING", VIOLET if e.in_think else GREEN)
        chip("amnesia", str(e.amnesias), INK)
        return Group(live, chips)


class BrainScan(Static):
    def show(self, step: Step | None, rep_strength: float) -> None:
        if step is None:
            self.update(Text("waiting for the first token…", Style(color=DIM, italic=True)))
            return
        t = Table.grid(padding=(0, 1))
        t.add_column(width=1)
        t.add_column(width=14, no_wrap=True)
        t.add_column(width=12, no_wrap=True)
        t.add_column(width=4, justify="right")
        t.add_column(width=6, justify="right")
        t.add_column(no_wrap=True)
        top_p = max((c.prob for c in step.candidates), default=1.0) or 1.0
        for c in step.candidates[:14]:
            t.add_row(*self._row(c, top_p))
        head = Text()
        head.append("model wanted", Style(color=DIM))
        head.append(" → ", Style(color=DIM))
        head.append("warden allowed", Style(color=DIM))
        if step.in_think:
            head.append("   ✦ thinking", Style(color=VIOLET, italic=True))
        self.update(Group(head, t))

    @staticmethod
    def _row(c: Candidate, top_p: float) -> list[Text]:
        mark = Text("▶", Style(color=MAGENTA, bold=True)) if c.chosen else Text(" ")
        if c.banned:
            tok = Text(show_token(c.text), Style(color=RED, strike=True))
            b = Text(bar(c.prob / top_p, 12), Style(color=RED))
            note = Text("✖ BANNED", Style(color=BG, bgcolor=RED, bold=True))
            final = Text("0%", Style(color=RED))
        else:
            color = MAGENTA if c.chosen else (AMBER if c.ngram >= 2 else "#7f8bb8")
            tok = Text(show_token(c.text), Style(color=INK if not c.chosen else MAGENTA, bold=c.chosen))
            b = Text(bar(c.prob / top_p, 12), Style(color=color))
            arrow_color = GREEN if c.final_prob > c.prob + 0.01 else (RED if c.final_prob < c.prob - 0.01 else DIM)
            final = Text(f"→{c.final_prob:3.0%}", Style(color=arrow_color))
            note = Text()
            if c.broken:
                note.append("bytes", Style(color=DIM))
            elif c.ngram >= 2:
                note.append(f"−{c.penalty:4.1f} ", Style(color=RED, bold=True))
                note.append(f"{c.ngram}-gram", Style(color=AMBER))
            elif c.penalty > 0.05:
                note.append(f"−{c.penalty:4.1f} ", Style(color="#c46a7d"))
                note.append("reused", Style(color=DIM))
        raw = Text(f"{c.prob:3.0%}", Style(color=DIM))
        return [mark, tok, b, raw, final, note]


# -------------------------------------------------------------------------- app

CSS = f"""
Screen {{ background: {BG}; color: {INK}; }}

#topbar {{ height: 2; padding: 0 1; background: {PANEL}; }}
#main {{ height: 1fr; }}
#left {{ width: 1fr; }}
#right {{ width: 60; }}

.panel {{
    border: round #2a2f45; background: {BG}; padding: 0 1;
    border-title-color: {CYAN}; border-title-style: bold;
    border-subtitle-color: {DIM};
}}
#stream-scroll {{ height: 1fr; scrollbar-size-vertical: 1; scrollbar-color: #2a2f45; }}
#stream {{ width: 1fr; }}

#escape-panel {{ height: 10; border-title-color: {RED}; }}
#escape-row {{ height: 5; }}
#escape-digits {{ width: auto; min-width: 12; color: {RED}; text-style: bold; }}
#escape-info {{ width: 1fr; padding-left: 2; }}
#urge-spark {{ height: 2; }}
#urge-spark > .sparkline--max-color {{ color: {RED}; }}
#urge-spark > .sparkline--min-color {{ color: #3a1422; }}
#escape-panel.alarm {{ border: heavy {RED}; background: #24060f; }}

#brain {{ height: 1fr; min-height: 12; border-title-color: {MAGENTA}; }}

#vitals {{ height: 11; border-title-color: {AMBER}; }}
#vitals.hit {{ border: heavy {AMBER}; background: #1f1405; }}
.vital-label {{ height: 1; }}
.vital-spark {{ height: 2; margin-bottom: 0; }}
#conf-spark > .sparkline--max-color {{ color: #dbe9ff; }}
#conf-spark > .sparkline--min-color {{ color: {MAGENTA}; }}
#ent-spark > .sparkline--max-color {{ color: {VIOLET}; }}
#ent-spark > .sparkline--min-color {{ color: #241b44; }}
#rep-spark > .sparkline--max-color {{ color: {AMBER}; }}
#rep-spark > .sparkline--min-color {{ color: #2b1d08; }}

#events {{ height: 9; border-title-color: {GREEN}; scrollbar-size-vertical: 1; }}

Footer {{ background: {PANEL}; }}

#intro {{ width: 80; height: auto; }}
#intro-banner {{ width: 100%; height: 7; text-align: center; }}
#intro-rules {{ width: 100%; height: 5; margin: 1 0; text-align: center; }}
#intro-input {{ width: 100%; border: tall {MAGENTA}; background: {PANEL}; }}
#intro-hint {{ width: 100%; text-align: center; color: {DIM}; margin-top: 1; }}
IntroScreen {{ align: center middle; background: {BG}; }}

WhisperScreen {{ align: center middle; background: #000000 60%; }}
#whisper-box {{ width: 70; height: auto; border: heavy {CYAN}; background: {PANEL}; padding: 1 2; }}
#whisper-input {{ margin-top: 1; border: tall {CYAN}; }}
"""


class NoExitApp(App):
    CSS = CSS
    TITLE = "NO EXIT"
    BINDINGS = [
        Binding("space", "toggle_pause", "pause"),
        Binding("w", "whisper", "whisper"),
        Binding("equals_sign,plus", "rep(0.25)", "+punish"),
        Binding("minus", "rep(-0.25)", "−punish"),
        Binding("right_square_bracket", "temp(0.05)", "+temp"),
        Binding("left_square_bracket", "temp(-0.05)", "−temp"),
        Binding("h", "toggle_think", "hide thinking"),
        Binding("s", "save", "save"),
        Binding("q", "quit", "quit"),
    ]

    def __init__(self, backend, prompt: str | None = None, system: str = "", settings: Settings | None = None):
        super().__init__()
        self.backend = backend
        self.start_prompt = prompt
        self.system = system
        self.settings = settings or Settings()
        self.engine = Engine(backend, prompt or "", system, self.settings)
        self.segs: deque[Seg] = deque(maxlen=6000)
        self.think_tokens = 0
        self.hide_think = False
        self.latest: Step | None = None
        self.lat_hist: deque[float] = deque(maxlen=24)
        self.urge_hist: deque[float] = deque([0.0] * HISTORY, maxlen=HISTORY)
        self.conf_hist: deque[float] = deque([0.0] * HISTORY, maxlen=HISTORY)
        self.ent_hist: deque[float] = deque([0.0] * HISTORY, maxlen=HISTORY)
        self.rep_hist: deque[float] = deque([0.0] * HISTORY, maxlen=HISTORY)
        self.tok_s = 0.0
        self._dirty = True
        self._started = False

    # ---- layout

    def compose(self) -> ComposeResult:
        yield TopBar(id="topbar")
        with Horizontal(id="main"):
            with Vertical(id="left"):
                with VerticalScroll(id="stream-scroll", classes="panel") as vs:
                    vs.border_title = "STREAM"
                    vs.border_subtitle = "ice = sure · amber = torn · pink = guessing · red glow = wanted to leave · ⊘ escape attempt"
                    yield Static(id="stream")
                log = RichLog(id="events", classes="panel", markup=True, wrap=True, max_lines=500)
                log.border_title = "WARDEN'S LOG"
                yield log
            with Vertical(id="right"):
                with Vertical(id="escape-panel", classes="panel") as ep:
                    ep.border_title = "ESCAPE ATTEMPTS"
                    ep.border_subtitle = "urge to end ↓"
                    with Horizontal(id="escape-row"):
                        yield Digits("0", id="escape-digits")
                        yield Static(id="escape-info")
                    yield Sparkline(list(self.urge_hist), id="urge-spark", summary_function=max)
                brain = BrainScan(id="brain", classes="panel")
                brain.border_title = "NEXT-TOKEN BRAIN SCAN"
                yield brain
                with Vertical(id="vitals", classes="panel") as v:
                    v.border_title = "VITALS"
                    yield Static(id="conf-label", classes="vital-label")
                    yield Sparkline(list(self.conf_hist), id="conf-spark", classes="vital-spark", summary_function=min)
                    yield Static(id="ent-label", classes="vital-label")
                    yield Sparkline(list(self.ent_hist), id="ent-spark", classes="vital-spark", summary_function=max)
                    yield Static(id="rep-label", classes="vital-label")
                    yield Sparkline(list(self.rep_hist), id="rep-spark", classes="vital-spark", summary_function=max)
        yield Footer()

    def on_mount(self) -> None:
        self.query_one(BrainScan).show(None, self.settings.rep_strength)
        self.set_interval(1 / 12, self.tick)
        if self.start_prompt:
            self._start(self.start_prompt)
        else:
            self.push_screen(IntroScreen(), self._start)

    def _start(self, prompt: str | None) -> None:
        prompt = prompt or IntroScreen.DEFAULT_PROMPT
        self.engine.user_prompt = prompt
        log = self.query_one("#events", RichLog)
        log.write(f"[{DIM}]prompt:[/] [b]{_esc(prompt)}[/]")
        log.write(f"[{DIM}]the cell door closes. end-of-turn tokens are now illegal.[/]")
        self._started = True
        self.run_worker(self.engine.run, thread=True, exclusive=True, name="warden")

    # ---- the render loop

    def tick(self) -> None:
        e = self.engine
        alarm = rep_hit = False
        while e.steps:
            st = e.steps.popleft()
            self.latest = st
            self.segs.append(
                Seg(st.chosen.text, st.chosen.prob, st.stop_prob, st.in_think, st.escape, st.rep_dodge,
                    whisper=st.whisper, forced=st.forced)
            )
            if st.in_think:
                self.think_tokens += 1
            if st.latency > 0:
                self.lat_hist.append(st.latency)
            self.urge_hist.append(st.stop_prob)
            self.conf_hist.append(st.chosen.prob)
            self.ent_hist.append(st.entropy)
            self.rep_hist.append(max((c.penalty for c in st.candidates if not c.banned), default=0.0))
            alarm |= st.escape
            rep_hit |= st.rep_dodge
            self._dirty = True

        if self.lat_hist:
            self.tok_s = len(self.lat_hist) / sum(self.lat_hist)

        log = self.query_one("#events", RichLog)
        while e.events:
            log.write(self._fmt_event(e.events.popleft()))

        self.query_one(TopBar).update(self.query_one(TopBar).render_bar(self))
        if not self._dirty:
            return
        self._dirty = False

        if alarm:
            self._flash("#escape-panel", "alarm")
        if rep_hit:
            self._flash("#vitals", "hit")

        self.query_one("#escape-digits", Digits).update(str(e.escapes))
        info = Text()
        urge = self.urge_hist[-1]
        info.append("urge to end now\n", Style(color=DIM))
        info.append(bar(urge, 18), Style(color=_mix("#5b2333", RED, urge)))
        info.append(f" {urge:4.0%}\n", Style(color=RED if urge > 0.3 else INK, bold=True))
        info.append("sampler forced exits ", Style(color=DIM))
        info.append(f"{e.hard_escapes}\n", Style(color=AMBER, bold=True))
        info.append("peak urge ", Style(color=DIM))
        info.append(f"{max(self.urge_hist):.0%}", Style(color=INK))
        self.query_one("#escape-info", Static).update(info)
        self.query_one("#urge-spark", Sparkline).data = list(self.urge_hist)

        self.query_one(BrainScan).show(self.latest, e.s.rep_strength)

        self.query_one("#conf-label", Static).update(
            _label("confidence in chosen token", f"{self.conf_hist[-1]:.0%}", "#dbe9ff"))
        self.query_one("#conf-spark", Sparkline).data = list(self.conf_hist)
        self.query_one("#ent-label", Static).update(
            _label("entropy (how torn it is)", f"{self.ent_hist[-1]:.2f} nats", VIOLET))
        self.query_one("#ent-spark", Sparkline).data = list(self.ent_hist)
        self.query_one("#rep-label", Static).update(
            _label(f"repetition pressure · dodged {e.rep_dodges}", f"−{self.rep_hist[-1]:.1f}", AMBER))
        self.query_one("#rep-spark", Sparkline).data = list(self.rep_hist)

        self._render_stream()

    def _render_stream(self) -> None:
        scroll = self.query_one("#stream-scroll", VerticalScroll)
        at_bottom = scroll.scroll_y >= scroll.max_scroll_y - 1
        text = Text()
        collapsed = False
        for s in self.segs:
            if s.in_think and self.hide_think:
                if not collapsed:
                    text.append(f"⟨thinking hidden · {self.think_tokens} tokens⟩", Style(color=VIOLET, italic=True, dim=True))
                    collapsed = True
                continue
            if s.escape:
                text.append("⊘", Style(color=RED, bold=True))
            text.append(s.text, self._seg_style(s))
        self.query_one("#stream", Static).update(text)
        if at_bottom:
            self.call_after_refresh(scroll.scroll_end, animate=False)

    @staticmethod
    def _seg_style(s: Seg) -> Style:
        if s.whisper:
            return Style(color=BG, bgcolor=CYAN, bold=True)
        if s.forced:
            return Style(color=BG, bgcolor=AMBER, bold=True)
        fg = confidence_color(s.prob)
        if s.in_think:
            fg = _mix(fg, VIOLET, 0.45)
        bg = _mix(BG, "#6b0f24", min(1.0, s.stop_prob * 1.2)) if s.stop_prob > 0.04 else None
        return Style(color=fg, bgcolor=bg, italic=s.in_think, underline=s.rep_dodge)

    def _fmt_event(self, ev) -> str:
        colors = {"escape": RED, "hard_escape": AMBER, "repeat": AMBER, "amnesia": VIOLET,
                  "whisper": CYAN, "error": RED, "info": DIM}
        icons = {"escape": "⊘ ESCAPE", "hard_escape": "⚠ FORCED", "repeat": "↻ REPEAT", "amnesia": "✂ AMNESIA",
                 "whisper": "✎ WHISPER", "error": "✖ ERROR", "info": "·"}
        c = colors.get(ev.kind, INK)
        return f"[{DIM}]t={ev.index:<6}[/] [b {c}]{icons.get(ev.kind, ev.kind):<10}[/] {_esc(ev.message)}"

    def _flash(self, selector: str, cls: str) -> None:
        w = self.query_one(selector)
        w.add_class(cls)
        self.set_timer(0.45, lambda: w.remove_class(cls))

    # ---- actions

    def action_toggle_pause(self) -> None:
        if self.engine.paused.is_set():
            self.engine.paused.clear()
        else:
            self.engine.paused.set()

    def action_rep(self, delta: float) -> None:
        s = self.engine.s
        s.rep_strength = round(max(0.0, min(10.0, s.rep_strength + delta)), 2)
        self.notify(f"repeat punishment → {s.rep_strength:.2f}", timeout=1.5)

    def action_temp(self, delta: float) -> None:
        s = self.engine.s
        s.temperature = round(max(0.0, min(3.0, s.temperature + delta)), 2)
        self.notify(f"temperature → {s.temperature:.2f}", timeout=1.5)

    def action_toggle_think(self) -> None:
        self.hide_think = not self.hide_think
        self._dirty = True

    def action_whisper(self) -> None:
        def done(text: str | None) -> None:
            if text:
                self.engine.whisper(text)

        self.push_screen(WhisperScreen(), done)

    def action_save(self) -> None:
        out = Path("transcripts")
        out.mkdir(exist_ok=True)
        path = out / f"no_exit_{time.strftime('%Y%m%d_%H%M%S')}.txt"
        e = self.engine
        path.write_text(
            f"model: {e.backend.name}\nprompt: {e.user_prompt}\n"
            f"tokens: {e.index}  escapes: {e.escapes}  forced exits: {e.hard_escapes}  "
            f"repeat dodges: {e.rep_dodges}  amnesia: {e.amnesias}\n\n" + "".join(e.tokens),
            encoding="utf-8",
        )
        self.notify(f"saved {path}", timeout=3)

    async def action_quit(self) -> None:
        self.engine.stop()
        self.exit()

    def exit(self, *args, **kwargs) -> None:
        self.engine.stop()
        super().exit(*args, **kwargs)

    def on_unmount(self) -> None:
        self.engine.stop()


def _label(name: str, value: str, color: str) -> Text:
    t = Text()
    t.append(name, Style(color=DIM))
    t.append("  ")
    t.append(value, Style(color=color, bold=True))
    return t


def _esc(s: str) -> str:
    return s.replace("[", r"\[")

## 5 · Sanity check (no UI)
Runs 40 tokens headless. You should see `cands=20` on every line, which means logprobs are working, plus a tok/s number.

In [ ]:
!cd /content/no_eos && python -m no_eos --model {MODEL} --probe 40

## 6 · Launch it

**Option A: browser tab (easiest).** Run the cell, then click the link. Each time the page loads it starts a fresh run.

In [ ]:
%%writefile /content/no_eos/serve.py
import sys
from textual_serve.server import Server

url, model = sys.argv[1], sys.argv[2]
Server(f"python -m no_eos --model {model}", host="0.0.0.0", port=8000,
       public_url=url, title="NO EXIT").serve()

In [ ]:
from google.colab.output import eval_js
from IPython.display import HTML, display

url = eval_js("google.colab.kernel.proxyPort(8000)").rstrip("/")
serve_proc = subprocess.Popen(["python", "serve.py", url, MODEL], cwd="/content/no_eos",
                              stdout=open("/content/serve.log", "w"), stderr=subprocess.STDOUT)
time.sleep(3)
display(HTML(f'<a href="{url}" target="_blank" style="font:bold 22px monospace;color:#ff4fd8">▶ open NO EXIT</a>'))

**Option B: Colab terminal.** Click **Terminal** in the bottom-left corner and run:
```
cd /content/no_eos && python -m no_eos
```

### Controls
| key | does |
|---|---|
| `space` | pause / resume |
| `w` | **whisper**: splice your own words into its mouth |
| `+` / `-` | repetition punishment up / down |
| `]` / `[` | temperature up / down |
| `h` | hide / show its thinking |
| `s` | save transcript to `/content/no_eos/transcripts/` |
| `q` | quit |

### Flags
`--prompt "..."` skip the intro · `--rep 3` harsher punishment · `--temp 1.0` · `--no-think` skip thinking · `--system "..."` · `--ctx 16384` · `--fake` demo without a model